# Big Data Analytics Homework 02

*Complete this assignment in Google Colab. Prior to submitting a copy of this notebook (.ipynb format), run every cell and ensure you have corrected all runtime errors. Be sure to fill in your Name and SUID in the following cell. As always, you must do your own work. This means you may not use answers to the following questions generated by any other person or a generative AI tool such as ChatGPT. You may, however, discuss this assignment with others in a general way and seek help when you need it, but, again, you must do your own work.*

Name:

SUID:

## Setup

In [1]:
! pip install pyspark -q

In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkContext
from pyspark.sql import functions as fn

sc = SparkContext.getOrCreate()

spark = SparkSession\
    .builder\
    .appName('Homework 02')\
    .getOrCreate()

## RDDs

### Q1

Create a single-dimensional PySpark RDD named `bernoulli_rdd` that contains 1,000 Bernoulli probability distribution data points consisting of integers 0 or 1 with P(0) = P(1) = 0.5.

Use only PySpark RDDs and `map` to complete this question.

Hint: Use the [`RandomRDDs.uniformRDD`](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.mllib.random.RandomRDDs.html#pyspark.mllib.random.RandomRDDs.uniformRDD) function for your sampling step. But you will still need to map over this RDD with a custom function that creates 1's and 0's from the initial output.

In [3]:
# your code here
from pyspark.mllib.random import RandomRDDs
#Create a uniform distribution since RandomRDDs does not contain a Bernoulli distribution
uniform_rdd=RandomRDDs.uniformRDD(sc,1000)
#Use map function to map uniform distribution to a Bernoulli distribution using lambda instead of def a function
bernoulli_rdd=uniform_rdd.map(lambda x:1 if x <0.5 else 0)

In [4]:
# do not modify
bernoulli_rdd.take(15)

[0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1]

### Q2

Using only Spark, count and display the number of 1's and the number of 0's in `bernoulli_rdd`.

Your output must be of the form "There are X 1's and Y 0's in bernoulli_rdd".

In [5]:
# your code here
#Create variable to store the counts of each value in the distribution
dist_counts=bernoulli_rdd.countByValue()
#Make a small loop to iterate through the counts and display the totals for each
for value, count in dist_counts.items():
  print(f"Value {value}: {count} occurrences")

Value 0: 485 occurrences
Value 1: 515 occurrences


### Q3

Create a two new **2-dimensional** RDDs named `bernoulli_sample_rdd_1` and `bernoulli_sample_rdd_2` that each contain sample data from `bernoulli_rdd`.

Each element of these RDDs should contain 10 samples (with replacement) from the original 1,000. Each RDD should contain 50 elements. In addition to the samples themselves, each data element in each RDD should contain `sample_size`, which should be calculated from each sample, not "hard-coded" as 10.

A single element of the result will be of the form `[(sample_size, [sample])]`.

`bernoulli_sample_rdd_1` should be created using the [`sample`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.sample.html) method and `bernoulli_sample_rdd_2` should be created using the[`takeSample`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.takeSample.html) method.

In [6]:
# your code here
#bernoulli_sample_rdd_1

#With Replacement using sample method
#Creating a variable to store dynamic sample size
sample_size=int(bernoulli_rdd.count()*0.01)
#Defining Function
def sample_with_replacement(rdd, sample_size, total_elements=50):
  #create an empty element for storage of sampled observations
  sampled_rdd=[]
  #for loop to iterate through bernoulli_rdd and take sample wr
  for r in range(total_elements):
    sample=rdd.sample(withReplacement=True, fraction=sample_size).take(sample_size)
    sampled_rdd.append(sample)
  return sampled_rdd

#Now we can create the 2D RDD
bernoulli_sample_rdd_1 = sc.parallelize(sample_with_replacement(bernoulli_rdd, sample_size))


In [7]:
#bernoulli_sample_rdd_2
#Creating a variable to store dynamic sample size
sample_size2=int(bernoulli_rdd.count()*0.01)
#Defining Function
def sample_with_replacement2(rdd, sample_size, total_elements=50):
  #create an empty element for storage of sampled observations
  sampled_rdd2=[]
  #for loop to iterate through bernoulli_rdd and take sample wr
  for r in range(total_elements):
    sample=rdd.takeSample(withReplacement=True, num=sample_size)
    sampled_rdd2.append(sample)
  return sampled_rdd2

#Now we can create the 2D RDD
bernoulli_sample_rdd_2 = sc.parallelize(sample_with_replacement2(bernoulli_rdd, sample_size))

In [8]:
# do not modify
bernoulli_sample_rdd_1.take(10)

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [9]:
# do not modify
bernoulli_sample_rdd_2.take(10)

[[0, 0, 1, 1, 0, 0, 1, 0, 1, 0],
 [0, 0, 1, 0, 1, 1, 0, 0, 0, 0],
 [1, 0, 0, 1, 0, 1, 1, 1, 0, 1],
 [1, 0, 1, 1, 1, 1, 1, 1, 1, 1],
 [0, 1, 1, 1, 0, 0, 1, 1, 1, 1],
 [1, 0, 0, 1, 1, 0, 0, 0, 1, 0],
 [0, 1, 1, 1, 0, 1, 0, 1, 1, 0],
 [0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
 [1, 1, 1, 0, 0, 1, 0, 0, 0, 0],
 [1, 1, 0, 0, 1, 1, 0, 0, 0, 0]]

### Q4

Explain the **key difference** between `bernoulli_sample_rdd_1` and `bernoulli_sample_rdd_2` and the reason for it.

The primary diffence between the two methods is that the sample method samples the RDD and creates a new RDD based on the parameters. The takeSample on the other hand samples the RDD and creates a list which can then be futher manupilated with or without Spark. Additionally, the samples displayed between the two differ due to the randomization of the sampling method used.

### Q5

Re-using code from above that created `bernoulli_sample_rdd_2`, create `bernoulli_sample_rdd_3` which has 100 observations per sample.

Using PySpark `map`, create a new RDD named `bernoulli_sample_mean_rdd` that contains the sampling distribution of the means of the samples contained in `bernoulli_sample_rdd_3`.

In [10]:
# your code here
#bernoulli_sample_rdd_3
#Creating a variable to store dynamic sample size
sample_size3=int(bernoulli_rdd.count()*0.10)
#Defining Function
def sample_with_replacement3(rdd, sample_size, total_elements=50):
  #create an empty element for storage of sampled observations
  sampled_rdd3=[]
  #for loop to iterate through bernoulli_rdd and take sample wr
  for r in range(total_elements):
    sample=rdd.takeSample(withReplacement=True, num=sample_size)
    sampled_rdd3.append(sample)
  return sampled_rdd3

#Now we can create the 2D RDD
bernoulli_sample_rdd_3 = sc.parallelize(sample_with_replacement3(bernoulli_rdd, sample_size))

In [11]:
#Create the bernoulli_sample_mean_rdd rdd and store the distrbution of the means of the samples
#Do this using map function, use lambda to iterate through bernoulli_sample_rdd_3
#sum/len provides the mean for each sample
bernoulli_sample_mean_rdd=bernoulli_sample_rdd_3.map(lambda sample: sum(sample)/ len(sample))

In [12]:
# do not modify
bernoulli_sample_mean_rdd.take(10)

[0.3, 0.6, 0.4, 0.6, 0.6, 0.9, 0.6, 0.8, 0.7, 0.7]

## DataFrames

In this section we will work with data from the U.S. Environmental Protection Agency (EPA). There are two data sets. The first data set consists of daily **temperatures** collected at the U.S. **city** level. The second data set consists of daily **air quality** data collected at the U.S. **county** level. These measurements were taken for the full year 2021.

In [14]:
# download the temperature and aqi data sets
%%bash

if [[ ! -f us-daily-temperatures-2021.csv.csv ]]; then
 wget https://syr-bda.s3.us-east-2.amazonaws.com/us-daily-temperatures-2021.csv -q
fi

if [[ ! -f us-daily-aqi-2021.csv.csv ]]; then
 wget https://syr-bda.s3.us-east-2.amazonaws.com/us-daily-aqi-2021.csv -q
fi

### Q6

Load the temperature data (using Spark) into a data frame called `temperature`. Load the air quality data into a data frame called `aqi`. Print the schema and the number of rows for each data set.

In [16]:
# your code here
#point to the file paths individully via new variable
temp_file="/content/us-daily-temperatures-2021.csv"
air_file="/content/us-daily-aqi-2021.csv"

#Load in the files to dataframes
temperature=spark.read.csv(temp_file, header=True, inferSchema=True)
aqi=spark.read.csv(air_file, header=True, inferSchema=True)

#Print the Schema and the number of rows for each dataframe
temperature.printSchema()
tempr_row_count=temperature.count()
print(f"Number of rows in DF: {tempr_row_count}")

aqi.printSchema()
air_row_count=aqi.count()
print(f"Number of rows in DF: {air_row_count}")


root
 |-- date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- city: string (nullable = true)
 |-- sites_reporting: integer (nullable = true)
 |-- mean_temperature_f: double (nullable = true)
 |-- max_temp_f: double (nullable = true)
 |-- max_temp_hour: integer (nullable = true)

Number of rows in DF: 206292
root
 |-- date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- aqi: integer (nullable = true)
 |-- category: string (nullable = true)

Number of rows in DF: 130922


In [15]:
!ls /content

sample_data  us-daily-aqi-2021.csv  us-daily-temperatures-2021.csv


The `temperature` data is reported at the city level, but the `aqi` data is at the county level. We want the "grain" of these data sets to match. This means we need to aggregate the the `temperature` data to the county level. This is tricky because we could possibly have counties in different states with the same name. This means you'll want to aggregate not just at the county level (by date), but at the state *and* county level (by date). We also have 4 different metrics to deal with: the total number of sites, a mean, a max, and one value — `max_temp_hour` — that corresponds to the value of another metric, `max_temp_f`.

### Q7

Create a new data frame called `temperature_county` that contains the **mean** temperature, the **max** temperature, and the **total** sites reporting, for each unique `date`, `state`, and `county` in the `temperature` data frame. The columns `mean_temperature_f`, `max_temp_f`, and `sites_reporting` should retain their names. Additionally, round the **mean** and **max** columns to the nearest whole number and cast them as integers.

In [17]:
# your code
#import functions from pyspark to use groupby and aggregate functions
from pyspark.sql import functions as f

#create new dataframe adhering to listed constraints
temperature_county=(
    #Grouping the data by the date, state, and county as specified
    temperature.groupBy("date", "state", "county")
    #Mean temp rounded and cast to int for use in new dataframe
    .agg(f.round(f.avg("mean_temperature_f"), 0).cast("int").alias("mean_temperature_f"),
    #Max temps rounded and cast to integer for use in new dataframe
    f.round(f.max("max_temp_f"), 0).cast("int").alias("max_temp_f"),
    f.sum("sites_reporting").alias("sites_reporting")
         )
)

In [18]:
# do not modify
print('Rows in temperature_county:', temperature_county.count())
temperature_county.orderBy('date', 'state', 'county').show(10)

Rows in temperature_county: 138555
+----------+--------+--------------------+------------------+----------+---------------+
|      date|   state|              county|mean_temperature_f|max_temp_f|sites_reporting|
+----------+--------+--------------------+------------------+----------+---------------+
|2021-01-01| Alabama|            Escambia|                64|        69|              1|
|2021-01-01| Alabama|           Jefferson|                66|        74|              2|
|2021-01-01|  Alaska|              Denali|                -1|         6|              1|
|2021-01-01|  Alaska|Fairbanks North Star|                -9|        -1|              7|
|2021-01-01| Arizona|             Cochise|                40|        49|              1|
|2021-01-01| Arizona|            Coconino|                29|        35|              1|
|2021-01-01| Arizona|            Maricopa|                49|        65|              1|
|2021-01-01| Arizona|              Navajo|                30|        42|   

### Q8

Create a new data frame called `county_max_temp_hour` that reports the `max_temp_hour` at the same level of aggregation as `temperature_county` in the previous step. This means it should have the **same number of rows** as `temperature_county` and contain the same grouping fields, but only one metric, `max_temp_hour` — the hour at which the maximum temperature occured. Once you have created this data frame, **left join** it to `temperature county` as a new data frame called `temperature_county_final`.

I've provided some starter code for you below. Fill in where you see `???` in order to complete the answer.

In [20]:
# your code
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# define a window spec
window_spec = Window.partitionBy("date", "state", "county")\
  .orderBy(desc("max_temp_f"))

# add row number for each group
df = temperature\
  .withColumn('row_num', row_number()\
              .over(window_spec))

# filter to the first row using row_num
#Combined this step with the step below for one line of code and
#to prevent making an additional dataframe
#df2 = ???

# keep only the grouping columns and max_temp_hour
county_max_temp_hour = df.filter(df.row_num==1).select("date","state", "county", "max_temp_hour")

# join county_max_temp_hour to temperature_county
temperature_county_final = temperature_county.join(
    county_max_temp_hour,
    on=["date", "state", "county"],
    how='left'
)


In [21]:
# do not modify
print('Rows in county_max_temp_hour:', county_max_temp_hour.count())
county_max_temp_hour.orderBy('date', 'state', 'county').show(10)

Rows in county_max_temp_hour: 138555
+----------+--------+--------------------+-------------+
|      date|   state|              county|max_temp_hour|
+----------+--------+--------------------+-------------+
|2021-01-01| Alabama|            Escambia|            0|
|2021-01-01| Alabama|           Jefferson|           13|
|2021-01-01|  Alaska|              Denali|            4|
|2021-01-01|  Alaska|Fairbanks North Star|           12|
|2021-01-01| Arizona|             Cochise|           12|
|2021-01-01| Arizona|            Coconino|           12|
|2021-01-01| Arizona|            Maricopa|           13|
|2021-01-01| Arizona|              Navajo|           14|
|2021-01-01| Arizona|                Pima|           15|
|2021-01-01|Arkansas|             Pulaski|           10|
+----------+--------+--------------------+-------------+
only showing top 10 rows



In [22]:
# do not modify
print('Rows in temperature_county_final:', temperature_county_final.count())
temperature_county_final.orderBy('date', 'state', 'county').show(10)

Rows in temperature_county_final: 138555
+----------+--------+--------------------+------------------+----------+---------------+-------------+
|      date|   state|              county|mean_temperature_f|max_temp_f|sites_reporting|max_temp_hour|
+----------+--------+--------------------+------------------+----------+---------------+-------------+
|2021-01-01| Alabama|            Escambia|                64|        69|              1|            0|
|2021-01-01| Alabama|           Jefferson|                66|        74|              2|           13|
|2021-01-01|  Alaska|              Denali|                -1|         6|              1|            4|
|2021-01-01|  Alaska|Fairbanks North Star|                -9|        -1|              7|           12|
|2021-01-01| Arizona|             Cochise|                40|        49|              1|           12|
|2021-01-01| Arizona|            Coconino|                29|        35|              1|           12|
|2021-01-01| Arizona|           

### Q9

Join `aqi` to `temperature_county_final` and call the resulting data frame `daily_county_measurements`. The join should be such that `daily_county_measurements` has the same number of rows as `temperature_county_final`.

Then write code that produces the `date`, name of the `county`, `state`, and `aqi` value where the **highest recorded `aqi`** occurred in 2021. In the event of a tie, take the first instance. Your answer should take the following form:

"The highest recorded AQI value in 2021 occured on [date], in [county] County, [state], and had a value of [aqi]."

I have included some comments and starter code to help you out.

In [23]:
# your code here

# join aqi to temperature_county_final
# and call the resulting data frame daily_county_measurements
daily_county_measurements = temperature_county_final.join(
    aqi,
    on=["date", "state", "county"],
    how='left'
)

#Use window again to get ordered aqi
window_spec2= Window.orderBy(desc("aqi"))
#and index to create ranking for measurements in df
daily_county_measurements_ranked=daily_county_measurements.withColumn(
    "row_num", row_number().over(window_spec2)
)
# create a data frame with a single row
# that contains the highest recorded AQI value in 2021
highest_aqi = daily_county_measurements_ranked.filter(daily_county_measurements_ranked.row_num ==1)


# extract values from the data frame needed for printing
#date = ''.join(row['date'] for row in highest_aqi)
highest_aqi_values=highest_aqi.select("date", "county", "state", "aqi").first()

#Define values to be pulled
date=highest_aqi_values["date"]
county=highest_aqi_values["county"]
state=highest_aqi_values["state"]
aqi_val=highest_aqi_values["aqi"]

# print the output as specified
print(f"The Highest recorded AQI value in 2001 occured on {date}, in {county} County, {state}, and had a value of {aqi_val}")

The Highest recorded AQI value in 2001 occured on 2021-09-14, in Tulare County, California, and had a value of 537


### Q10

Using a process similar to Q8, create a new data frame called `highest_temperates_by_state_2021` that contains one row per state, and shows the `date`, `state`, and `max_temp_f` for the **highest recorded temperature** in that state in 2021. In the case of ties, pick the earliest day of the year.

In [25]:
# your code here
#Use window to create a partition on the data of interest
window_spec3=Window.partitionBy("state").orderBy(desc("max_temp_f"), f.asc("date"))
#use row number to assign a rank to the partitions
highest_temperates_by_state_2021=temperature\
    .withColumn("row_num", row_number().over(window_spec3))\
    .filter("row_num=1")\
    .select("date", "state", "max_temp_f")

In [26]:
# do not modify
highest_temperates_by_state_2021\
  .select('date', 'state', 'max_temp_f')\
  .orderBy(desc('max_temp_f'))\
  .show()

+----------+-----------------+----------+
|      date|            state|max_temp_f|
+----------+-----------------+----------+
|2021-11-20|       California|     129.0|
|2021-06-17|Country Of Mexico|     123.1|
|2021-06-17|          Arizona|     118.0|
|2021-07-10|           Nevada|     118.0|
|2021-06-28|           Oregon|     116.0|
|2021-06-29|       Washington|     116.0|
|2021-06-29|            Idaho|     115.6|
|2021-08-17|           Kansas|     115.0|
|2021-09-06|             Utah|     115.0|
|2021-07-29|      Mississippi|     110.1|
|2021-06-15|          Montana|     110.0|
|2021-06-12|       New Mexico|     110.0|
|2021-06-15|          Wyoming|     109.0|
|2021-06-11|            Texas|     107.0|
|2021-06-17|         Colorado|     105.0|
|2021-06-17|             Iowa|     104.0|
|2021-07-31|         Oklahoma|     103.9|
|2021-07-19|     North Dakota|     103.0|
|2021-06-17|         Nebraska|     102.4|
|2021-07-30|         Arkansas|     101.0|
+----------+-----------------+----